In [ ]:
"""
Description:
    CHMC Implementation with AVF: FPI
    USE THE CORRECT ENVIRONMENT:  CHMC_FALL_2025
    YYYY-MM-DD

Author: John Gallagher
Created: 2025-09-28
Last Modified: 2025-10-21
Version: 1.0.0

"""

import jax
import jax.numpy as jnp
from jax import jit
import time
from scipy.sparse import diags

jax.config.update("jax_enable_x64", True)


def gen_nnormal(dim = 2, precision_matrix=None, cov=None):
    
    class MultipleMatrices(Exception):
            pass
    if precision_matrix is not None and cov is not None:
        raise MultipleMatrices(
            "Please supply either a Precision Matrix or a Covariance Matrix"
        )
    if precision_matrix is None and cov is not None:
        precision_matrix = jnp.linalg.inv(cov)
    if precision_matrix is None and cov is None:
        precision_matrix = jnp.eye(dim)
        
    def nnormal(x):
        """n-Dim Gaussian target distribution."""
        return jnp.exp(-0.5 * (x @ precision_matrix @ x))
    return nnormal

def gen_perturb_precision(dim):
    prec = jnp.diag(jnp.ones(dim))
    prec += 0.05*jnp.diag(jnp.ones(dim-1), k=-1)
    prec += 0.05*jnp.diag(jnp.ones(dim-1), k=1)
    # L -= 0.1*jnp.tri(dim, k=-2)
    # prec_out = L @ L.T
    return prec

# May need determinant later. 

def qex(qp):
    """
    qex: q extracted from qp state vector
    """
    dim = len(qp) // 2
    return qp[:dim]


def pex(qp):
    """
    pex: p extracted from qp state vector
    """
    dim = len(qp) // 2
    return qp[dim:]


def J_sym(vec):
    """
    J is the symplectic Jacobian matrix for Hamiltonians where J = ([[0, I]])
    """
    dim = len(vec) // 2
    return jnp.concatenate([vec[dim:], -vec[:dim]])


def qJ_sym(vec):
    """
    updates q side of vector with qdot = p
    Returns
    array([p], [0])
    """
    dim = len(vec) // 2
    return jnp.concatenate([vec[dim:], jnp.zeros(dim)])


def pJ_sym(vec):
    """
    qp with p = -qdot only
    Returns
    array([p], [0])
    """
    dim = len(vec) // 2
    return jnp.concatenate([jnp.zeros(dim), -vec[:dim]])


def draw_p(qp, key):
    q = qex(qp)
    p = jax.random.normal(key, shape=(dim,))
    return jnp.concatenate([q, p]), None


def gen_leapfrog(gradH, tau, N):
    def leapfrog(qp):
        """
        Requires gradH, tau, N
        Leapfrog integrator
        Takes state vector qp, and integrates it according to hamiltonian Ham

        """

        def lf_step(carry_in, _):
            qp0 = carry_in
            qhalf_p0 = qp0 + 0.5 * tau * qJ_sym(gradH(qp0))
            qhalf_pout = qhalf_p0 + tau * pJ_sym(gradH(qhalf_p0))
            qp_out = qhalf_pout + 0.5 * tau * qJ_sym(gradH(qhalf_pout))
            return qp_out, _

        qp_final, _ = jax.lax.scan(lf_step, qp, xs=None, length=N)
        return qp_final

    return leapfrog

def gen_midpointFPI(gradH, tau, N, tol,maxIter, solve = jnp.linalg.solve):
    """
    Generates midpointFPI function with appropriate statics: 
    tau, tol, maxIter, 
    """
    def midpointFPI(qp, _):
        """
        FPI_mid integrator
        Requries qp:statevector, and gradH defined before hand

        y(i+1) = y(i) + tau * J_sym GradH( 0.5*(y(i)+y(i+1)))

        """
        x0 = qp

        def G(y):
            """
            G(y) = x0 + tau * J_sym GradH( 0.5*(x+y))
            """
            midpoint = 0.5 * (x0 + y)
            return x0 + tau * J_sym(gradH(midpoint))

        def F(y):
            return y - G(y)

        def newton_step(qp):
            jacF = jax.jacobian(F)
            qpout = x0 - solve(jacF(qp), F(qp))
            return qpout

        def cond(carry):
            i, qp = carry
            Fqp = F(qp)
            err = jnp.linalg.norm(Fqp)
            return (err > tol) & (i < maxIter)

        def body_step(carry):
            i, qp = carry
            return [i + 1, newton_step(qp)]

        _, qp_out = jax.lax.while_loop(cond, body_step, [0, qp])
        return qp_out, qp_out
    def midpointFPI_T(qp):
        qp_out, _ = jax.lax.scan(midpointFPI, qp, xs = None, length = N)
        return qp_out
    return midpointFPI_T

def accept(delta, key):
    alpha = jnp.minimum(1.0, jnp.exp(delta))
    u = jax.random.uniform(key, shape=())
    return u <= alpha


def gen_hmc_kernel(H, tau, N):
    gradH = jax.grad(H)
    integrator = gen_leapfrog(gradH, tau, N)

    def hmc_kernel(carry_in, key):
        carry, _, _ = carry_in
        qp0, _ = draw_p(carry, key)
        qp_star = integrator(qp0)
        deltaH =  H(qp_star) - H(qp0)  # -(final - init) = init -final
        is_accepted = accept(-deltaH, key)
        qp_out = jnp.where(is_accepted, qp_star, qp0)
        carry_out = [qp_out, deltaH, is_accepted]
        return carry_out, carry_out
    return hmc_kernel
def gen_chmc_kernel(H, tau, N, tol, maxIter, solve=jnp.linalg.solve):
    gradH = jax.grad(H)
    integrator = gen_midpointFPI(gradH, tau, N, tol, maxIter, solve=jnp.linalg.solve )

    def chmc_kernel(carry_in, key):
        carry, _, _ = carry_in
        qp0, _ = draw_p(carry, key)
        qp_star = integrator(qp0)
        deltaH = H(qp0) - H(qp_star)  # -(final - init) = init -final
        is_accepted = accept(deltaH, key)
        qp_out = jnp.where(is_accepted, qp_star, qp0)
        carry_out = [qp_out, deltaH, is_accepted]
        return carry_out, carry_out
    return chmc_kernel

def hmc_sampler(initial_sample, keys, H, tau, N):
    """
    inputs: initial_sample, keys, H, tau, T
    """
    hmc_kernel = gen_hmc_kernel(H, tau, N)
    _, samples = jax.lax.scan(hmc_kernel, initial_sample, xs=keys)
    return samples
def chmc_sampler(initial_sample, keys, H, tau, N, tol, maxIter, solve=jnp.linalg.solve):
    """
    inputs: initial_sample, keys, H, tau, N, tol, maxIter, solve=jnp.linalg.solve
    """
    chmc_kernel = gen_chmc_kernel(H, tau, N, tol, maxIter, solve=jnp.linalg.solve)
    _, samples = jax.lax.scan(chmc_kernel, initial_sample, xs=keys)
    return samples    

def gen_hamiltonian(Mass_inv, target):
    
    def hamiltonian(qp):
        q, p = qex(qp), pex(qp)
        return 0.5 * jnp.sum(p @ Mass_inv @ p) - jnp.log(target(q))
    return hamiltonian
def gen_hidim_hamiltonian(Mass_inv, Prec_mat):
    def hamiltonian(qp):
        q, p = qex(qp), pex(qp)
        return 0.5 * jnp.sum(p@Mass_inv@p) + 0.5* jnp.sum(q@Prec_mat@q)
    return hamiltonian
# def J_H(gH):
#     """Same operation as Symplectic Jacobian"""
#     return jnp.concatenate([gH[dim:], -gH[:dim]])


key = jax.random.PRNGKey(1)

dim = 2**8
Mass_inv = jnp.eye(dim)
# pert_precision = gen_perturb_precision(dim)
# target = gen_nnormal(precision_matrix=pert_precision)
target_mat = gen_perturb_precision(dim)
hamiltonian = gen_hidim_hamiltonian(Mass_inv, target_mat)
# target = gen_nnormal(dim)
# hamiltonian = gen_hamiltonian(Mass_inv, target)


## FOR HIGH CONDITION PRECISION MATRIX
# hamiltonian = gen_hicond_hamiltonian(Mass_inv, pert_precision)
jit_H = jit(hamiltonian)
gradH = jax.jit(jax.grad(hamiltonian))

# jit_integrator = jax.jit(leapfrog)
# jit_integrator = jit(midpointFPI)

# Set parameters


initnum_samples = 1
mainnum_samples = 100
keys_start = jax.random.split(key, initnum_samples)
keys_main = jax.random.split(key, mainnum_samples)
qp_init = jax.random.normal(key, shape=(2 * dim,))

# Structure of carry
# init_sample: [Array: sample, float: deltaH, bool: Accepted]
init_sample = [qp_init, 1, False]

tau = 0.05
T = 1
N = int(jnp.ceil(T/tau))
tol = 1e-3
maxIter = 2
# compile
start = time.time()
jhmc_sampler = jit(hmc_sampler, static_argnums=(2,3,4))
sample_hmc = jhmc_sampler(init_sample, keys_start, hamiltonian, tau, N)
end = time.time()
print("1st run:", end - start)
# main run
start = time.time()
sample_hmc = jhmc_sampler(init_sample, keys_main,  hamiltonian, tau, N)
end = time.time()
print(
    f"Main run:\n {mainnum_samples} runs: {end - start:.2f} \n 1 run:  {(end-start)/mainnum_samples}"
)
start = time.time()
jchmc_sampler = jit(chmc_sampler, static_argnums=(2,3,4,5,6))
sample_chmc = jchmc_sampler(init_sample, keys_start, hamiltonian, tau, N, tol, maxIter)
end = time.time()
print("CHMC 1st run:", end - start)
start = time.time()
sample_chmc = jchmc_sampler(init_sample, keys_main, hamiltonian, tau, N, tol, maxIter)
end = time.time()
print(
    f"CHMC Main run:\n {mainnum_samples} runs: {end - start:.2f} \n 1 run:  {(end-start)/mainnum_samples}"
)

## Tests on different precision matrices' condition

In [ ]:
import numpy as np
# def gen_perturb_precision(dim):
#     L = jnp.diag(jnp.ones(dim))
#     L+= jnp.tri(2*jnp.ones(dim), k=-1)
#     prec_out = L @ L.T
#     return prec_out
def gen_perturb_precision2(dim):
    prec = jnp.diag(np.ones(dim))
    prec -= 2*jnp.diag(np.ones(dim-1), k=-1)
    prec -= 2*jnp.diag(np.ones(dim-1), k=1)
    # L -= 0.1*jnp.tri(dim, k=-2)
    # prec_out = L @ L.T
    return prec
testdim = 2**12
Mass_inv = jnp.eye(testdim)
hicond_prec = gen_perturb_precision(testdim)
midcond_prec = gen_perturb_precision2(testdim)
hicond_nnormal_jax = gen_nnormal(precision_matrix=hicond_prec)
midcond_nnormal_jax = gen_nnormal(precision_matrix=midcond_prec)
hicond_hamiltonian = gen_hamiltonian(Mass_inv, hicond_nnormal_jax)
midcond_hamiltonian = gen_hamiltonian(Mass_inv, midcond_nnormal_jax)

key = jax.random.PRNGKey(2)
x0 = jax.random.uniform(key, (2*testdim))
print('hiham:',hicond_hamiltonian(x0))
print('midham:',midcond_hamiltonian(x0))
print('prec1: ',np.linalg.cond(gen_perturb_precision(testdim)))
print('prec2:', np.linalg.cond(gen_perturb_precision2(testdim)))
q,p = qex(x0), pex(x0)
hicond_hamiltonian(x0)
midcond_nnormal_jax(q)
def gen_nnormal(precision_matrix=None, cov=None, dim = 2):
    class MultipleMatrices(Exception):
            pass
    if precision_matrix is not None and cov is not None:
        raise MultipleMatrices(
            "Please supply either a Precision Matrix or a Covariance Matrix"
        )
    if precision_matrix is None and cov is not None:
        precision_matrix = jnp.linalg.inv(cov)
    if precision_matrix is None and cov is None:
        precision_matrix = jnp.eye(dim)
    def nnormal(x):
        """n-Dim Gaussian target distribution."""
        return jnp.exp(-0.5 * (x @ precision_matrix @ x))
    return nnormal

hicond_nnormal_jax = gen_nnormal(precision_matrix=hicond_prec)
hicond_nnormal_jax(q)
def gen_highcond_hamiltonian(Mass_inv, Prec_mat):
    def hamiltonian(qp):
        q, p = qex(qp), pex(qp)
        return 0.5 * jnp.sum(p@Mass_inv@p) + 0.5* jnp.sum(q@Prec_mat@q)
    return hamiltonian
hiH = gen_highcond_hamiltonian(Mass_inv, hicond_prec)
hiH(x0)
gradhiH = jax.grad(hiH)
np.linalg.norm(gradhiH(x0))
# hicond_prec

## HMC For Loop $\tau$, For Loop $\verb|dims|$:

In [ ]:
import numpy as np

# Set parameters


# dims
ndims = 21
dims = np.logspace(2,12, ndims,base=2, dtype=int)

# taus to scan
T = 1
numtaus = 4
tauinit = 0.2
tau_set = tauinit*1/jnp.logspace(0,3,numtaus,base=2)

# initialize keys
key = jax.random.PRNGKey(1)
mainnum_samples = 1000
keys_start = jax.random.split(key, initnum_samples)
keys_main = jax.random.split(key, mainnum_samples)

# Structure of carry
# [Array: sample, float: deltaH, bool: Accepted]

# init_sample: [Array: sample, float: deltaH, bool: Accepted]


# initialize receiving arrays
samples_deltaHs = np.zeros((mainnum_samples,numtaus, ndims))
samples_accepted = np.zeros((mainnum_samples,numtaus, ndims))
hmc_samples = []
for dim in dims:
    hmc_samples.append(np.zeros((mainnum_samples, dim)))




jhmc_sampler = jit(hmc_sampler, static_argnums=(2,3,4))
for i, taus in enumerate(tau_set):
    for j, dim in enumerate(dims):
        N = int(jnp.ceil(T/taus))
        # if dim == 4:
        if dim == 1024:
            print(f'taus: {taus}, dim: {dim}, N: {N}')
        Mass_inv = jnp.eye(dim)

        # target = gen_nnormal(dim)
        # hamiltonian = gen_hamiltonian(Mass_inv, target)
        
        
        # use gen_hidim_hamiltonian to avoid numerical issues with e^-300 = 0 => divide by zero issue with grad. 
        target_mat = gen_perturb_precision(dim)
        hamiltonian = gen_hidim_hamiltonian(Mass_inv, target_mat)

        jit_H = jit(hamiltonian)
        gradH = jax.jit(jax.grad(hamiltonian))
        qp_init = jax.random.normal(key, shape=(2 * dim,))
        init_sample = [qp_init, 1, False]
        hmc_samples[j], samples_deltaHs[:,i, j], samples_accepted[:,i, j] = hmc_sampler(init_sample, keys_main,  hamiltonian, taus, N)
        print(f'tau: {tau_set[i]:.3f}, dim = {dims[j]}\n  #Accepts: {samples_accepted[:,i,j].sum()}')
        if dim > 700:
            print(f'Saving Samples: t{taus}_d{dim}')
            np.save(f'hmc_samples/hmc_samples_t{taus}_d{dim}.npy', hmc_samples[i])
            np.save(f'hmc_samples_deltaHs/hmc_samples_deltaHs_t{taus}_d{dim}.npy', samples_deltaHs[:,i, j])
            np.save(f'hmc_samples_accepted/hmc_samples_accepted_t{taus}_d{dim}.npy', samples_accepted[:,i, j])

In [ ]:
# import os
# # os.makedirs('chmc_samples')
# os.makedirs('chmc_samples_deltaHs')
# os.makedirs('chmc_samples_accepted')

## CHMC $\tau = 0.2$ forloop dims:

In [ ]:
import numpy as np
##### Previous run parameters

ndims = 16
dims = np.logspace(2,9.612, ndims,base=2, dtype=int)

# # taus to scan
# T = 1
# numtaus = 4
# tauinit = 0.2
# tau_set = tauinit*1/jnp.logspace(0,3,numtaus,base=2)
taus = 0.2
tol = 1e-3
maxIter = 2
# initialize keys
key = jax.random.PRNGKey(1)
mainnum_samples = 1000
keys_start = jax.random.split(key, initnum_samples)
keys_main = jax.random.split(key, mainnum_samples)

chmc_samples_deltaHs = np.zeros((mainnum_samples, ndims))
chmc_samples_accepted = np.zeros((mainnum_samples, ndims))
chmc_samples = []
for dim in dims:
    chmc_samples.append(np.zeros((mainnum_samples, dim)))

jchmc_sampler = jit(chmc_sampler, static_argnums=(2,3,4,5,6))
# for i, taus in enumerate(tau_set):
for j, dim in enumerate(dims):
    N = int(jnp.ceil(T/taus))
    # if dim == 4:
    # if dim == 1024:
    print(f'taus: {taus}, dim: {dim}, N: {N}')
    start = time.time()
    Mass_inv = jnp.eye(dim)

    # target = gen_nnormal(dim)
    # hamiltonian = gen_hamiltonian(Mass_inv, target)
    
    
    # use gen_hidim_hamiltonian to avoid numerical issues with e^-300 = 0 => divide by zero issue with grad. 
    target_mat = gen_perturb_precision(dim)
    hamiltonian = gen_hidim_hamiltonian(Mass_inv, target_mat)

    jit_H = jit(hamiltonian)
    gradH = jax.jit(jax.grad(hamiltonian))
    qp_init = jax.random.normal(key, shape=(2 * dim,))
    init_sample = [qp_init, 1, False]
    chmc_samples[j], chmc_samples_deltaHs[:, j], chmc_samples_accepted[:, j] = chmc_sampler(init_sample, keys_main,  hamiltonian, taus, N, tol, maxIter)
    end = time.time()
    print(f"CHMC Main run: Dim: {dim}\n {mainnum_samples} runs: {end - start:.2f}, 1 run:  {(end-start)/mainnum_samples}")
    print(f'tau: {taus:.3f}, dim = {dims[j]}\n  #Accepts: {chmc_samples_accepted[:,j].sum()}')
    if dim > 4000:
        print(f'Saving Samples: t{taus}_d{dim}')
        np.save(f'chmc_samples/chmc_samples_t{taus}_d{dim}.npy', chmc_samples[j])
        np.save(f'chmc_samples_deltaHs/chmc_samples_deltaHs_t{taus}_d{dim}.npy', chmc_samples_deltaHs[:, j])
        np.save(f'chmc_samples_accepted/chmc_samples_accepted_t{taus}_d{dim}.npy', chmc_samples_accepted[:, j])

#### Forgot one dimension so I just re-ran it

In [ ]:
# dim = 786
# taus = 0.2
# tol = 1e-3
# maxIter = 2
# T = 1
# N = int(jnp.ceil(T/taus))
# key = jax.random.PRNGKey(1)
# mainnum_samples = 1000
# keys_start = jax.random.split(key, initnum_samples)
# keys_main = jax.random.split(key, mainnum_samples)

# chmc_786_deltaHs = np.zeros((mainnum_samples, ndims))
# chmc_786_accepted = np.zeros((mainnum_samples, ndims))
# # if dim == 4:
# # if dim == 1024:
# print(f'taus: {taus}, dim: {dim}, N: {N}')
# start = time.time()
# Mass_inv = jnp.eye(dim)

# # target = gen_nnormal(dim)
# # hamiltonian = gen_hamiltonian(Mass_inv, target)


# # use gen_hidim_hamiltonian to avoid numerical issues with e^-300 = 0 => divide by zero issue with grad. 
# target_mat = gen_perturb_precision(dim)
# hamiltonian = gen_hidim_hamiltonian(Mass_inv, target_mat)

# jit_H = jit(hamiltonian)
# gradH = jax.jit(jax.grad(hamiltonian))
# qp_init = jax.random.normal(key, shape=(2 * dim,))
# init_sample = [qp_init, 1, False]

# chmc_786_samp, chmc_786_deltaH, chmc_786_accept = chmc_sampler(init_sample, keys_main,  hamiltonian, taus, N, tol, maxIter) 

### Loading/Saving the data

In [ ]:
# Saving the samples
# for j, dim in enumerate(dims):
#     print(f'Saving Samples: t{taus}_d{dim}')
#     np.save(f'chmc_samples/chmc_samples_t{taus}_d{dim}.npy', chmc_samples[j])
#     np.save(f'chmc_samples_deltaHs/chmc_samples_deltaHs_t{taus}_d{dim}.npy', chmc_samples_deltaHs[:, j])
#     np.save(f'chmc_samples_accepted/chmc_samples_accepted_t{taus}_d{dim}.npy', chmc_samples_accepted[:, j])
ndims = 5
dims10_12 = np.logspace(10,12, ndims,base=2, dtype=int)
for j, dim in enumerate(dims10_12):
    # print(dim)
    new_accept_col = np.load(f'chmc_samples_accepted/chmc_samples_accepted_t0.2_d{dim}.npy').reshape(-1,1)
    new_deltaHs_col = np.load(f'chmc_samples_deltaHs/chmc_samples_deltaHs_t0.2_d{dim}.npy').reshape(-1,1)
    chmc_samples_accepted = np.concatenate((chmc_samples_accepted,new_accept_col), axis = 1)
    chmc_samples_deltaHs = np.concatenate((chmc_samples_deltaHs, new_deltaHs_col),axis=1)
chmc_dims = np.logspace(2,9.612, ndims,base=2, dtype=int)
chmc_dims = np.concatenate((chmc_dims, dims10_12))

In [ ]:
np.logspace(2,12,21,base=2, dtype=int)

In [ ]:
# #### SAVING PREVIOUS NUMERICAL EXPERIMENT: 

# dims
ndims = 22
dims = np.logspace(2,12, ndims,base=2, dtype=int)

# taus to scan
T = 1
numtaus = 4
tauinit = 0.2
tau_set = tauinit*1/jnp.logspace(0,3,numtaus,base=2)

##### SAVE ######
# for i, taus in enumerate(tau_set):
#     for j, dim in enumerate(dims):
#         np.save(f'hmc_samples/hmc_samples_t{taus}_d{dim}.npy', hmc_samples[i])
#         np.save(f'hmc_samples_deltaHs/hmc_samples_deltaHs_t{taus}_d{dim}.npy', samples_deltaHs[:,i, j])
#         np.save(f'hmc_samples_accepted/hmc_samples_accepted_t{taus}_d{dim}.npy', samples_accepted[:,i, j])


##### LOAD #####

samples_deltaHs = np.zeros((mainnum_samples,numtaus, ndims))
for i, taus in enumerate(tau_set):
    for j, dim in enumerate(dims):
        # np.load(f'hmc_samples/hmc_samples_t{taus}_d{dim}.npy', hmc_samples[i])
        samples_deltaHs[:,i, j] = np.load(f'hmc_samples_deltaHs/hmc_samples_deltaHs_t{taus}_d{dim}.npy')
        # np.load(f'hmc_samples_accepted/hmc_samples_accepted_t{taus}_d{dim}.npy', samples_accepted[:,i, j])

In [ ]:
# #### SAVING PREVIOUS NUMERICAL EXPERIMENT: 
# # ndims = 9
# # dims = np.logspace(4,12, ndims,base=2, dtype=int)
# # numtaus = 4
# # taufinal =0.5
# # tauinit = 0.1
# # tau_set = 0.2*1/jnp.logspace(0,3,numtaus,base=2)
# for i, taus in enumerate(tau_set):
#     for j, dim in enumerate(dims):
#         np.save(f'hmc_samples/hmc_samples_t{taus}_d{dim}.npy', hmc_samples[i])
#         np.save(f'hmc_samples_deltaHs/hmc_samples_deltaHs_t{taus}_d{dim}.npy', samples_deltaHs[:,i, j])
#         np.save(f'hmc_samples_accepted/hmc_samples_accepted_t{taus}_d{dim}.npy', samples_accepted[:,i, j])

In [ ]:
#### SAVING PREVIOUS NUMERICAL EXPERIMENT: 
# ndims = 7
# dims = np.logspace(4,10, ndims,base=2, dtype=int)
# numtaus = 7
# taufinal =0.5a
# tauinit = 0.1
# tau_set = 0.2*1/jnp.logspace(0,3,numtaus,base=2)
# for i in range(len(hmc_samples)):
#     np.save(f'hmc_samples_7taus_2_10_dims/hmc_samples_7taus_2_10_dims_{i}.npy', hmc_samples[i])
# np.save('hmc_samples_7taus_2_10_dims/hmc_samples_detlaHs_7taus_2_10_dims.npy', samples_deltaHs)
# np.save('hmc_samples_7taus_2_10_dims/hmc_samples_accepted_7taus_2_10_dims.npy', samples_accepted)

In [ ]:
jnp.linalg.norm(q@midcond_prec@q)

## Vectorized inner loop for $\tau, N$

In [ ]:
import numpy as np
ndims = 9
dims = np.logspace(2,10, ndims,base=2, dtype=int)
# Mass_inv = jnp.eye(dim)
# target = gauss_ndimf_jax
# hamiltonian = gen_hamiltonian(Mass_inv, target)
# grad_target = jit(jax.grad(target))
# jit_H = jit(hamiltonian)
# gradH = jax.jit(jax.grad(hamiltonian))

# jit_integrator = jax.jit(leapfrog)
# jit_integrator = jit(midpointFPI)

# Set parameters
key = jax.random.PRNGKey(1)

initnum_samples = 1
mainnum_samples = 1000
keys_start = jax.random.split(key, initnum_samples)
keys_main = jax.random.split(key, mainnum_samples)
# qp_init = jax.random.normal(key, shape=(2 * dim,))

# Structure of carry
# init_sample: [Array: sample, float: deltaH, bool: Accepted]

tol = 1e-4
max_iter = 100
tau = 0.2
T = 1

# tol = 1e-4
# max_iter = 100
numtaus = 7
taufinal =0.5
tauinit = 0.1
tau_set = 0.2*1/jnp.logspace(0,6,numtaus,base=2)
Ns = T/tau_set

# tau_set = jnp.linspace(tauinit, taufinal, numtaus)
# [Array: sample, float: deltaH, bool: Accepted]
# samples_taus = np.zeros((mainnum_samples, 2*dim, numtaus, ndims))
# (7,1000) into shape (1000,7)
samples_deltaHs = np.zeros((numtaus, ndims, mainnum_samples))
samples_accepted = np.zeros((numtaus, ndims, mainnum_samples))
hmc_samples = []
for dim in dims:
    hmc_samples.append(np.zeros((dim, mainnum_samples)))


# initial_sample, keys, H, tau, N
vhmc_sampler = jax.vmap(hmc_sampler, in_axes=(None, None, None, 0, None))

jhmc_sampler = jit(hmc_sampler, static_argnums=(2,3,4))
for j, dim in enumerate(dims):
    N = int(jnp.ceil(T/taus))
    # if dim == 4:
    print(f'dim: {dim}, N: {N}')
    Mass_inv = jnp.eye(dim)
    target = gauss_ndimf_jax
    hamiltonian = gen_hamiltonian(Mass_inv, target)
    grad_target = jit(jax.grad(target))
    jit_H = jit(hamiltonian)
    gradH = jax.jit(jax.grad(hamiltonian))
    qp_init = jax.random.normal(key, shape=(2 * dim,))
    init_sample = [qp_init, 1, False]
    hmc_samples[:], samples_deltaHs[:,j,: ], samples_accepted[:,j, :] = vhmc_sampler(init_sample, keys_main,  hamiltonian, tau_set, N)

In [ ]:
numtaus = 5
taufinal =0.5
tauinit = 0.1
dtau = (taufinal-tauinit)/(numtaus -1)
print(dtau)

In [ ]:
print(f'(num samples, num tau, numdims)\n {samples_deltaHs.shape}')
for i in range(len(tau_set)):
    for j in range(len(dims)):
        # print(f'tau: {tau_set[i]:.3f}, dim = {dims[j]}\n min: {samples_deltaHs[:,i,j].min():.6f} max: {samples_deltaHs[:,i,j].max():.6f} mean: {samples_deltaHs[:,i,j].mean():.7f} std: {samples_deltaHs[:,i,j].std():.6f} #Accepts: {samples_accepted[:,i,j].sum()}')
        print(f'tau: {tau_set[i]:.3f}, dim = {dims[j]}\n  #Accepts: {samples_accepted[:,i,j].sum()}')

In [ ]:
print(samples_deltaHs.shape)
print(tau_set.shape, dims.shape)

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["text.usetex"] = True
fig,axs = plt.subplots(len(tau_set))
fig.set_size_inches(8,20)

# index: (mainnum_samples,numtaus, ndims)
dtau = (taufinal-tauinit)/(numtaus -1)

for i in range(len(tau_set)):
    for j in range(len(dims)):
        axs[i].hist(samples_deltaHs[i,:,j],bins=10, density = True, label = f'{dims[j]}-dim')
        axs[i].set_xlim(-0.5, 0.5)
        axs[i].set_ylabel
        axs[i].set_title(f'$\\tau$: {tau_set[i]:0.5f}, T={T}  $\Longrightarrow$   N = {int(jnp.ceil(T/((i+1)*dtau)))}')
        axs[i].legend()
fig.suptitle('Histogram of LF: $\Delta H$', y=0.91)


In [ ]:
def alphaex(deltaH):
    return jnp.minimum(1., jnp.exp(deltaH))
valphaex = jax.vmap(alphaex)
alphas = alphaex(samples_deltaHs)
meanalphas = alphas.mean(axis=1)
print(meanalphas.shape)
# print(meanalphas)
print(len(dims), len(meanalphas[0,:]))


In [ ]:
chmc_alphas = alphaex(chmc_samples_deltaHs)
chmc_alphas.mean(axis=0)

In [ ]:
chmc_dims

In [ ]:
import matplotlib.pyplot as plt

chmc_dims = np.logspace(2,12,21,base=2, dtype=int)
def alphaex(deltaH):
    return jnp.minimum(1., jnp.exp(deltaH))
valphaex = jax.vmap(alphaex)
alphas = alphaex(samples_deltaHs)
meanalphas = alphas.mean(axis=0)
# meanalphas.shape
# intersection = np.interp(dims,dims, meanalphas)
chmc_alphas = alphaex(chmc_samples_deltaHs)
chmc_meanalphas = chmc_alphas.mean(axis=0)
sixteens = np.logspace(2,12,11,base=2)
for onetau in range(len(tau_set)):
    plt.semilogx(dims, meanalphas[onetau,:], marker = '.', label = f'$\\tau  = {tau_set[onetau]:.5f}$', base = 10)
# plt.semilogx(sixteens, np.ones(len(sixteens))*0.965, base=16,marker = '*', color ='red')

plt.semilogx(chmc_dims, chmc_meanalphas, marker = '*', label = f'$\\tau = 0.2$ CHMC', base=16)
plt.grid(True, which="major", ls="-", color='gray', alpha=0.5)
plt.grid(True, which="minor", ls=":", color='lightgray', alpha=0.4)
plt.ylabel(r'Mean$(\alpha)$')
plt.xlabel('Dimension')
plt.legend()
plt.title('Mean Acceptance Rate vs Dimension')
# plt.savefig('figures/LF_accept(tau)_v_dim_FINAL_base16.png')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

chmc_dims = np.logspace(2, 12, 21, base=2, dtype=int)

def alphaex(deltaH):
    return np.minimum(1., np.exp(deltaH))

valphaex = np.vectorize(alphaex)

# Compute mean and standard error for main data
alphas = alphaex(samples_deltaHs)
meanalphas = alphas.mean(axis=0)
stderrs = alphas.std(axis=0) / np.sqrt(alphas.shape[0])  # Standard error

# Compute mean and standard error for CHMC data
chmc_alphas = alphaex(chmc_samples_deltaHs)
chmc_meanalphas = chmc_alphas.mean(axis=0)
chmc_stderrs = chmc_alphas.std(axis=0) / np.sqrt(chmc_alphas.shape[0])  # Standard error
markers = ['o', 's', '^', 'v', 'D', 'P', 'X', '*', 'h', '8']
# Create the plot
plt.figure(figsize=(9, 5))
plt.rcParams['text.usetex'] = True
plt.rcParams['text.latex.preamble'] = r'\usepackage{lmodern}'
# Plot main data with error bars
for onetau in range(len(tau_set)):
    plt.errorbar(dims, meanalphas[onetau,:], 
                 yerr=stderrs[onetau,:], 
                 marker=markers[onetau], 
                 color = '#1f77b4',
                 capsize=3,
                 label=f'$\\tau = {tau_set[onetau]:.3f}$  (HMC-LF)')

# Plot CHMC data with error bars
plt.errorbar(chmc_dims, chmc_meanalphas, 
             yerr=chmc_stderrs, 
             marker='D', 
             capsize=5,
             color='red',
             label='$\\tau = 0.2$ (CHMC-IMP)')

plt.loglog(base=10)
plt.grid(True, which="major", ls="-", color='gray', alpha=0.5)
plt.grid(True, which="minor", ls=":", color='lightgray', alpha=0.4)
plt.ylabel(r'Mean$(\alpha)$', fontsize=22, labelpad=-1) 
plt.xlabel('$d$', fontsize = 22, labelpad=0)
plt.xticks(fontsize=18)
plt.xlim(4)
plt.tick_params(axis='y', which='minor', size=5)
plt.tick_params(axis='both', which='minor', labelsize=16)

# Format y-axis to show only 2 decimal places with size 15
def percent_formatter(x, pos):
    return f'${x*100:.0f}\\%$'

plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(percent_formatter))
plt.gca().yaxis.set_minor_formatter(plt.FuncFormatter(percent_formatter))
plt.tick_params(axis='y', labelsize=15)
plt.yticks(fontsize=16)
plt.legend(fontsize = 16)
plt.title('$\\textbf{Mean Acceptance Rate vs Dimension}$ ',fontsize = 24, fontweight='bold', color='#002856')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import LogFormatter

chmc_dims = np.logspace(2, 12, 21, base=2, dtype=int)

def alphaex(deltaH):
    return np.minimum(1., np.exp(deltaH))

valphaex = np.vectorize(alphaex)

# Compute mean and standard error for main data
alphas = alphaex(samples_deltaHs)
meanalphas = alphas.mean(axis=0)
stderrs = alphas.std(axis=0) / np.sqrt(alphas.shape[0])  # Standard error

# Compute mean and standard error for CHMC data
chmc_alphas = alphaex(chmc_samples_deltaHs)
chmc_meanalphas = chmc_alphas.mean(axis=0)
chmc_stderrs = chmc_alphas.std(axis=0) / np.sqrt(chmc_alphas.shape[0])  # Standard error

# Create the plot
plt.figure(figsize=(10, 6))

# Plot main data with error bars
for onetau in range(len(tau_set)):
    # Calculate upper and lower bounds for error bars in log space
    y_lower = np.maximum(meanalphas[onetau,:] - stderrs[onetau,:], 1e-6)  # Avoid negative values for log scale
    y_upper = meanalphas[onetau,:] + stderrs[onetau,:]
    
    plt.errorbar(dims, meanalphas[onetau,:], 
                 yerr=[meanalphas[onetau,:] - y_lower, y_upper - meanalphas[onetau,:]], 
                 marker='.', 
                 color = '#1f77b4',
                 capsize=3,
                 label=f'$\\tau = {tau_set[onetau]:.3f} LF$')

# Plot CHMC data with error bars
# Calculate upper and lower bounds for CHMC error bars in log space
chmc_y_lower = np.maximum(chmc_meanalphas - chmc_stderrs, 1e-6)  # Avoid negative values for log scale
chmc_y_upper = chmc_meanalphas + chmc_stderrs

plt.errorbar(chmc_dims, chmc_meanalphas, 
             yerr=[chmc_meanalphas - chmc_y_lower, chmc_y_upper - chmc_meanalphas], 
             marker='*', 
             capsize=5,
             color='red',
             label='$\\tau = 0.2$ CHMC')

# Set log scale for both axes
plt.xscale('log')
plt.yscale('log')

plt.grid(True, which="major", ls="-", color='gray', alpha=0.5)
plt.grid(True, which="minor", ls=":", color='lightgray', alpha=0.4)
plt.ylabel(r'Mean$(\alpha)$', fontsize=24)
plt.xlabel('Dimension', fontsize=24)

# Create a custom formatter that shows decimal values instead of scientific notation
class CustomLogFormatter(LogFormatter):
    def __call__(self, x, pos=None):
        if x > 0:
            return f'{x:.2f}'
        else:
            return ''

# Apply the custom formatter to both major and minor ticks on y-axis
formatter = CustomLogFormatter()
plt.gca().yaxis.set_major_formatter(formatter)
plt.gca().yaxis.set_minor_formatter(formatter)

plt.xticks(fontsize=15)
plt.yticks(fontsize=15)
plt.tick_params(axis='y', which='minor', size = 5, reset = True)
plt.tick_params(axis='both', which='minor', labelsize=15)
plt.tick_params(axis='both', which='major', labelsize=15)
plt.legend(fontsize=16)
plt.title('Mean Acceptance Rate vs Dimension', fontsize=30)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["text.usetex"] = True
fig,axs = plt.subplots(5)
fig.set_size_inches(8,20)

# index: (mainnum_samples,numtaus, ndims)
dtau = (taufinal-tauinit)/(numtaus -1)

for i in range(numtaus):
    for j in range(ndims):
        axs[i].hist(samples_deltaHs[:,i,j],bins=10, density = True, label = f'{2**(j+2)}-dim')
        axs[i].set_xlim(-0.5, 0.5)
        axs[i].set_ylabel
        axs[i].set_title(f'$\\tau$: {(i+1)*dtau:0.1f}, T={T}  $\Longrightarrow$   N = {int(jnp.ceil(T/((i+1)*dtau)))}')
        axs[i].legend()
fig.suptitle('Histogram of LF: $\Delta H$', y=0.91)


In [ ]:
import numpy as np
ndims = 5
dims = np.logspace(2,6, ndims,base=2, dtype=int)
# Mass_inv = jnp.eye(dim)
# target = gauss_ndimf_jax
# hamiltonian = gen_hamiltonian(Mass_inv, target)
# grad_target = jit(jax.grad(target))
# jit_H = jit(hamiltonian)
# gradH = jax.jit(jax.grad(hamiltonian))

# jit_integrator = jax.jit(leapfrog)
# jit_integrator = jit(midpointFPI)

# Set parameters
key = jax.random.PRNGKey(1)

initnum_samples = 1
mainnum_samples = 10000
keys_start = jax.random.split(key, initnum_samples)
keys_main = jax.random.split(key, mainnum_samples)
# qp_init = jax.random.normal(key, shape=(2 * dim,))

# Structure of carry
# init_sample: [Array: sample, float: deltaH, bool: Accepted]
init_sample = [qp_init, 1, False]

tol = 1e-4
maxIter = 1
tau = 0.2
T = 1

# tol = 1e-4
# maxIter = 100
numtaus = 5
taufinal =0.4
tauinit = 0.1
tau_set = jnp.linspace(tauinit, taufinal, numtaus)
# [Array: sample, float: deltaH, bool: Accepted]
# samples_taus = np.zeros((mainnum_samples, 2*dim, numtaus, ndims))
chmc_deltaHs = np.zeros((mainnum_samples,numtaus, ndims))
chmc_accepted = np.zeros((mainnum_samples,numtaus, ndims))
chmc_samples = []
for dim in dims:
    chmc_samples.append(np.zeros((mainnum_samples, dim)))

jchmc_sampler = jit(chmc_sampler, static_argnums=(2,3,4,5,6))
for i, taus in enumerate(tau_set):
    for j, dim in enumerate(dims):
        N = int(jnp.ceil(T/taus))
        if dim == 4:
            print(f'taus: {taus}, dim: {dim}, N: {N}')
        Mass_inv = jnp.eye(dim)
        target = gauss_ndimf_jax
        hamiltonian = gen_hamiltonian(Mass_inv, target)
        grad_target = jit(jax.grad(target))
        jit_H = jit(hamiltonian)
        gradH = jax.jit(jax.grad(hamiltonian))
        qp_init = jax.random.normal(key, shape=(2 * dim,))
        init_sample = [qp_init, 1, False]
        chmc_samples[j], chmc_deltaHs[:,i, j], chmc_accepted[:,i, j] = chmc_sampler(init_sample, keys_main,  hamiltonian, taus, N, tol, maxIter)
        # initial_sample, keys, H, tau, N, tol, maxIter, solve=jnp.linalg.solve

In [ ]:
print(f'(num samples, num tau, numdims)\n {chmc_deltaHs.shape}')
for i in range(numtaus):
    for j in range(ndims):
        print(f'tau: {(i+1)*0.1:.3f}, dim = {2**(j+1)}\n min: {chmc_deltaHs[:,i,j].min():.6f} max: {chmc_deltaHs[:,i,j].max():.6f} mean: {chmc_deltaHs[:,i,j].mean():.7f} std: {chmc_deltaHs[:,i,j].std():.6f} #Accepts: {chmc_accepted[:,i,j].sum()}')

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["text.usetex"] = True
fig,axs = plt.subplots(5)
fig.set_size_inches(8,20)

# index: (mainnum_samples,numtaus, ndims)
dtau = (taufinal-tauinit)/(numtaus -1)

for i in range(numtaus):
    for j in range(ndims):
        axs[i].hist(chmc_deltaHs[:,i,j],bins=10, density = True, label = f'{2**(j+1)}-dim')
        axs[i].set_xlim(-0.5, 0.5)
        axs[i].set_ylabel
        axs[i].set_title(f'$\\tau$: {(i+1)*dtau:0.1f}, T={T}  $\Longrightarrow$   N = {int(jnp.ceil(T/((i+1)*dtau)))}')
        axs[i].legend()
fig.suptitle('Histogram of LF: $\Delta H$', y=0.91)


In [ ]:
plt.hist(chmc_samples[0][:,2],bins=50,density=True)

In [ ]:
plt.hist2d(chmc_samples[0][:,2], chmc_samples[0][:,1],bins=25)

In [ ]:
plt.hist2d(chmc_samples[0][:,0], chmc_samples[0][:,1],bins=25)

In [ ]:
def scatter_hist(x, y, ax, ax_histx, ax_histy):
    # no labels
    ax_histx.tick_params(axis="x", labelbottom=False)
    ax_histy.tick_params(axis="y", labelleft=False)

    # the scatter plot:
    ax.scatter(x, y)

    # now determine nice limits by hand:
    binwidth = 0.25
    xymax = max(np.max(np.abs(x)), np.max(np.abs(y)))
    lim = (int(xymax/binwidth) + 1) * binwidth

    bins = np.arange(-lim, lim + binwidth, binwidth)
    ax_histx.hist(x, bins=bins)
    ax_histy.hist(y, bins=bins, orientation='horizontal')